# Sonoran Cycles Supply Chain Analytics Dataset Generator

## Overview

This notebook generates a synthetic ERP dataset for **Sonoran Cycles**, a fictional premium mountain bike manufacturer headquartered in Tucson, Arizona.

The purpose of this dataset is to support a portfolio project demonstrating:

- Demand Planning
- Inventory Optimization
- Purchasing Analytics
- Supplier Performance
- SQL Analysis
- Power BI Dashboard Development

The generated data is intentionally realistic while remaining simplified enough for analysis and visualization.

---

### Dataset Files Generated

- products.csv
- suppliers.csv
- calendar.csv
- sales_orders.csv
- purchase_orders.csv
- inventory_snapshots.csv

In [1]:
import pandas as pd
import numpy as np

from datetime import datetime, timedelta
from pathlib import Path
import random

In [2]:
# Make dataset reproducible

RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

In [3]:
# Output location

DATA_FOLDER = Path("data")

DATA_FOLDER.mkdir(exist_ok=True)

print(DATA_FOLDER.resolve())

/Users/samuellettes/Desktop/Portfolio Project/Supply Chain Analyst/sonoran-cycles-analytics/data


In [4]:
COMPANY = "Sonoran Cycles"

START_DATE = "2022-01-01"
END_DATE = "2026-12-31"

WAREHOUSES = [
    "Tucson Distribution Center",
    "Columbus Distribution Center"
]

SALES_CHANNELS = [
    "Dealer",
    "Direct-to-Consumer"
]

REGIONS = [
    "West",
    "Southwest",
    "Midwest",
    "Southeast",
    "Northeast"
]

In [5]:
products = [
    {
        "model": "Catalina",
        "category": "Cross Country",
        "travel": 120,
        "msrp": 3299
    },
    {
        "model": "Rincon",
        "category": "Downcountry",
        "travel": 130,
        "msrp": 3999
    },
    {
        "model": "Sabino",
        "category": "Trail",
        "travel": 140,
        "msrp": 4599
    },
    {
        "model": "Romero",
        "category": "Aggressive Trail",
        "travel": 150,
        "msrp": 5199
    },
    {
        "model": "Oracle",
        "category": "Enduro",
        "travel": 165,
        "msrp": 6199
    },
    {
        "model": "Sky Island",
        "category": "eMTB",
        "travel": 160,
        "msrp": 7499
    },
    {
        "model": "Sonoita",
        "category": "Gravel",
        "travel": 0,
        "msrp": 3499
    }
]

In [6]:
sizes = ["XS", "S", "M", "L", "XL"]

colors = [
    "Ironwood",
    "Saguaro",
    "Sunset",
    "Granite",
    "Coyote",
    "Sky"
]


## Generate Product Master

Generate the complete Sonoran Cycles product catalog by creating one SKU for every combination of model, size, and color.

Each SKU represents a unique inventory item that can be sold, forecasted, and replenished independently.

In [7]:
# Additional product attributes

wheel_sizes = {
    "Cross Country": "29",
    "Downcountry": "29",
    "Trail": "29",
    "Aggressive Trail": "29",
    "Enduro": "Mixed",
    "eMTB": "Mixed",
    "Gravel": "700c"
}

frame_materials = {
    "Cross Country": "Carbon",
    "Downcountry": "Carbon",
    "Trail": "Carbon",
    "Aggressive Trail": "Carbon",
    "Enduro": "Carbon",
    "eMTB": "Carbon",
    "Gravel": "Aluminum"
}

launch_dates = {
    "Catalina":"2022-03-01",
    "Rincon":"2022-06-01",
    "Sabino":"2022-09-01",
    "Romero":"2023-03-01",
    "Oracle":"2023-06-01",
    "Sky Island":"2023-09-01",
    "Sonoita":"2022-04-01"
}

In [8]:
product_rows = []

product_id = 1001

for product in products:

    for size in sizes:

        for color in colors:

            product_rows.append({

                "product_id": product_id,

                "model_name": product["model"],

                "category": product["category"],

                "travel_mm": product["travel"],

                "wheel_size": wheel_sizes[product["category"]],

                "frame_material": frame_materials[product["category"]],

                "size": size,

                "color": color,

                "msrp": product["msrp"],

                "launch_date": launch_dates[product["model"]],

                "active_flag": True

            })

            product_id += 1

In [9]:
products_df = pd.DataFrame(product_rows)

products_df.head()

,product_id,model_name,category,travel_mm,wheel_size,frame_material,size,color,msrp,launch_date,active_flag
0,1001,Catalina,Cross Country,120,29,Carbon,XS,Ironwood,3299,2022-03-01,True
1,1002,Catalina,Cross Country,120,29,Carbon,XS,Saguaro,3299,2022-03-01,True
2,1003,Catalina,Cross Country,120,29,Carbon,XS,Sunset,3299,2022-03-01,True
3,1004,Catalina,Cross Country,120,29,Carbon,XS,Granite,3299,2022-03-01,True
4,1005,Catalina,Cross Country,120,29,Carbon,XS,Coyote,3299,2022-03-01,True


In [10]:
products_df.to_csv(DATA_FOLDER / "products.csv", index=False)

print("products.csv saved successfully.")

products.csv saved successfully.


In [11]:
gross_margins = {
    "Cross Country": 0.42,
    "Downcountry": 0.41,
    "Trail": 0.40,
    "Aggressive Trail": 0.39,
    "Enduro": 0.38,
    "eMTB": 0.35,
    "Gravel": 0.43
}

In [12]:
# Product family
products_df["product_family"] = products_df["category"].apply(
    lambda x: "Mountain Bike" if x != "Gravel" else "Gravel"
)

# Model year
products_df["model_year"] = products_df["launch_date"].str[:4].astype(int)

# Product status
products_df["status"] = "Active"

# Gross margin
products_df["gross_margin"] = products_df["category"].map(gross_margins)

# Standard cost
products_df["standard_cost"] = (
    products_df["msrp"] * (1 - products_df["gross_margin"])
).round(2)

In [13]:
products_df = products_df[
    [
        "product_id",
        "model_name",
        "product_family",
        "category",
        "model_year",
        "status",
        "travel_mm",
        "wheel_size",
        "frame_material",
        "size",
        "color",
        "msrp",
        "standard_cost",
        "gross_margin",
        "launch_date",
        "active_flag",
    ]
]

## Data Validation

Before generating additional tables, validate that the Product Master is complete and internally consistent.

This ensures downstream tables reference accurate master data.

In [14]:
# Basic dataset validation

print(f"Rows: {len(products_df)}")
print(f"Columns: {len(products_df.columns)}")
print()

print("Unique Models:", products_df["model_name"].nunique())
print("Unique Sizes:", products_df["size"].nunique())
print("Unique Colors:", products_df["color"].nunique())
print()

print("Duplicate Product IDs:", products_df["product_id"].duplicated().sum())
print("Missing Values:", products_df.isna().sum().sum())

Rows: 210
Columns: 16

Unique Models: 7
Unique Sizes: 5
Unique Colors: 6

Duplicate Product IDs: 0
Missing Values: 0


In [15]:
products_df.groupby("category").size()

category
Aggressive Trail    30
Cross Country       30
Downcountry         30
Enduro              30
Gravel              30
Trail               30
eMTB                30
dtype: int64

In [16]:
products_df[["model_name","msrp","standard_cost","gross_margin"]].head(10)

,model_name,msrp,standard_cost,gross_margin
0,Catalina,3299,1913.42,0.42
1,Catalina,3299,1913.42,0.42
2,Catalina,3299,1913.42,0.42
3,Catalina,3299,1913.42,0.42
4,Catalina,3299,1913.42,0.42
5,Catalina,3299,1913.42,0.42
6,Catalina,3299,1913.42,0.42
7,Catalina,3299,1913.42,0.42
8,Catalina,3299,1913.42,0.42
9,Catalina,3299,1913.42,0.42


# Generate Calendar Dimension

The calendar dimension provides time-based attributes used throughout the analytics model.

This table supports:

- Demand Planning
- Seasonality Analysis
- Inventory Planning
- Sales Reporting
- Promotional Analysis

Every transactional table references this calendar through the order date.

In [17]:
# Generate daily calendar

calendar_df = pd.DataFrame({
    "date": pd.date_range(
        start=START_DATE,
        end=END_DATE,
        freq="D"
    )
})

calendar_df.head()

,date
0,2022-01-01
1,2022-01-02
2,2022-01-03
3,2022-01-04
4,2022-01-05


In [18]:
calendar_df["year"] = calendar_df["date"].dt.year

calendar_df["quarter"] = (
    "Q" +
    calendar_df["date"].dt.quarter.astype(str)
)

calendar_df["month"] = calendar_df["date"].dt.month

calendar_df["month_name"] = (
    calendar_df["date"]
    .dt.strftime("%B")
)

calendar_df["week"] = (
    calendar_df["date"]
    .dt.isocalendar()
    .week
)

calendar_df["day"] = calendar_df["date"].dt.day

calendar_df["day_name"] = (
    calendar_df["date"]
    .dt.strftime("%A")
)

calendar_df["day_of_week"] = (
    calendar_df["date"]
    .dt.dayofweek
)

calendar_df.head()

,date,year,quarter,month,month_name,week,day,day_name,day_of_week
0,2022-01-01,2022,Q1,1,January,52,1,Saturday,5
1,2022-01-02,2022,Q1,1,January,52,2,Sunday,6
2,2022-01-03,2022,Q1,1,January,1,3,Monday,0
3,2022-01-04,2022,Q1,1,January,1,4,Tuesday,1
4,2022-01-05,2022,Q1,1,January,1,5,Wednesday,2


In [19]:
calendar_df["is_weekend"] = (
    calendar_df["day_of_week"] >= 5
)

In [20]:
def riding_season(month):

    if month in [3,4,5]:
        return "Spring"

    elif month in [6,7,8]:
        return "Summer"

    elif month in [9,10]:
        return "Fall"

    else:
        return "Winter"

calendar_df["season"] = (
    calendar_df["month"]
    .apply(riding_season)
)

In [21]:
calendar_df["is_peak_season"] = (
    calendar_df["month"]
    .isin([3,4,5,6])
)

In [22]:
monthly_demand = {
    1:0.70,
    2:0.90,
    3:1.20,
    4:1.50,
    5:1.60,
    6:1.40,
    7:1.20,
    8:1.00,
    9:1.05,
    10:0.85,
    11:0.75,
    12:0.90
}

calendar_df["demand_multiplier"] = (
    calendar_df["month"]
    .map(monthly_demand)
)

In [23]:
calendar_df["promotion"] = "None"

In [24]:
# Spring Sale

calendar_df.loc[
    (calendar_df["month"] == 3) &
    (calendar_df["day"] <= 15),
    "promotion"
] = "Spring Sale"


# Memorial Day

calendar_df.loc[
    (calendar_df["month"] == 5) &
    (calendar_df["day"] >= 25),
    "promotion"
] = "Memorial Day"


# Labor Day

calendar_df.loc[
    (calendar_df["month"] == 9) &
    (calendar_df["day"] <= 7),
    "promotion"
] = "Labor Day"


# Black Friday

calendar_df.loc[
    (calendar_df["month"] == 11) &
    (calendar_df["day"] >= 24) &
    (calendar_df["day"] <= 30),
    "promotion"
] = "Black Friday"

In [25]:
calendar_df["holiday"] = (
    calendar_df["promotion"] != "None"
)

In [26]:
calendar_df.head(15)

,date,year,quarter,month,month_name,week,day,day_name,day_of_week,is_weekend,season,is_peak_season,demand_multiplier,promotion,holiday
0,2022-01-01,2022,Q1,1,January,52,1,Saturday,5,True,Winter,False,0.7,None,False
1,2022-01-02,2022,Q1,1,January,52,2,Sunday,6,True,Winter,False,0.7,None,False
2,2022-01-03,2022,Q1,1,January,1,3,Monday,0,False,Winter,False,0.7,None,False
3,2022-01-04,2022,Q1,1,January,1,4,Tuesday,1,False,Winter,False,0.7,None,False
4,2022-01-05,2022,Q1,1,January,1,5,Wednesday,2,False,Winter,False,0.7,None,False
5,2022-01-06,2022,Q1,1,January,1,6,Thursday,3,False,Winter,False,0.7,None,False
6,2022-01-07,2022,Q1,1,January,1,7,Friday,4,False,Winter,False,0.7,None,False
7,2022-01-08,2022,Q1,1,January,1,8,Saturday,5,True,Winter,False,0.7,None,False
8,2022-01-09,2022,Q1,1,January,1,9,Sunday,6,True,Winter,False,0.7,None,False
9,2022-01-10,2022,Q1,1,January,2,10,Monday,0,False,Winter,False,0.7,None,False


In [27]:
calendar_df.to_csv(
    DATA_FOLDER / "calendar.csv",
    index=False
)

print("calendar.csv saved successfully!")

calendar.csv saved successfully!


In [28]:
customers = [

    {
        "customer_name":"Sonoran Cycles Online",
        "customer_type":"DTC",
        "region":"National",
        "state":"Online",
        "dealer_profile":"Direct",
        "dealer_tier":"N/A"
    },

    {
        "customer_name":"Desert Trail Bikes",
        "customer_type":"Dealer",
        "region":"Southwest",
        "state":"Arizona",
        "dealer_profile":"Trail Shop",
        "dealer_tier":"Platinum"
    },

    {
        "customer_name":"Cactus Cycling",
        "customer_type":"Dealer",
        "region":"Southwest",
        "state":"Arizona",
        "dealer_profile":"Generalist",
        "dealer_tier":"Gold"
    },

    {
        "customer_name":"Sonoran Bike Co.",
        "customer_type":"Dealer",
        "region":"Southwest",
        "state":"Arizona",
        "dealer_profile":"Aggressive MTB",
        "dealer_tier":"Gold"
    },

    {
        "customer_name":"Mesa Mountain Bikes",
        "customer_type":"Dealer",
        "region":"Southwest",
        "state":"Arizona",
        "dealer_profile":"XC Specialist",
        "dealer_tier":"Silver"
    },

]

In [29]:
dealer_data = [

    # Direct-to-Consumer
    ("Sonoran Cycles Online", "DTC", "National", "Online", "Direct", "N/A"),

    # Southwest
    ("Desert Trail Bikes", "Dealer", "Southwest", "Arizona", "Trail Shop", "Platinum"),
    ("Cactus Cycling", "Dealer", "Southwest", "Arizona", "Generalist", "Gold"),
    ("Sonoran Bike Co.", "Dealer", "Southwest", "Arizona", "Aggressive MTB", "Gold"),
    ("Mesa Mountain Bikes", "Dealer", "Southwest", "Arizona", "XC Specialist", "Silver"),
    ("Tucson Trail Works", "Dealer", "Southwest", "Arizona", "Trail Shop", "Gold"),
    ("Red Rock Cycles", "Dealer", "Southwest", "Utah", "Aggressive MTB", "Silver"),
    ("Sedona Ride Center", "Dealer", "Southwest", "Arizona", "Premium Dealer", "Gold"),
    ("Las Cruces Cycling", "Dealer", "Southwest", "New Mexico", "Generalist", "Silver"),

    # West Coast
    ("Evergreen Cyclery", "Dealer", "West Coast", "Washington", "XC Specialist", "Platinum"),
    ("Cascade Bikes", "Dealer", "West Coast", "Washington", "Trail Shop", "Gold"),
    ("Rain City Cycles", "Dealer", "West Coast", "Washington", "Generalist", "Silver"),
    ("Pacific Trail Bikes", "Dealer", "West Coast", "California", "Premium Dealer", "Platinum"),
    ("Redwood Ride Shop", "Dealer", "West Coast", "California", "Generalist", "Gold"),
    ("Sierra Cycle Works", "Dealer", "West Coast", "California", "Trail Shop", "Gold"),
    ("Coastal Gravel Co.", "Dealer", "West Coast", "California", "Gravel Shop", "Silver"),
    ("Bay Mountain Bikes", "Dealer", "West Coast", "California", "Aggressive MTB", "Gold"),

    # Mountain
    ("High Country Bikes", "Dealer", "Mountain", "Colorado", "Aggressive MTB", "Platinum"),
    ("Front Range Cyclery", "Dealer", "Mountain", "Colorado", "Trail Shop", "Gold"),
    ("Alpine Ride Co.", "Dealer", "Mountain", "Colorado", "XC Specialist", "Gold"),
    ("Rocky Peak Cycles", "Dealer", "Mountain", "Colorado", "Generalist", "Silver"),
    ("Wasatch Wheel Works", "Dealer", "Mountain", "Utah", "Trail Shop", "Gold"),
    ("Bozeman Bike Works", "Dealer", "Mountain", "Montana", "Aggressive MTB", "Silver"),
    ("Teton Cycling", "Dealer", "Mountain", "Wyoming", "Generalist", "Silver"),

    # Midwest
    ("Lakeshore Bikes", "Dealer", "Midwest", "Wisconsin", "XC Specialist", "Gold"),
    ("Great Lakes Cyclery", "Dealer", "Midwest", "Michigan", "Generalist", "Silver"),
    ("Prairie Trail Bikes", "Dealer", "Midwest", "Minnesota", "Trail Shop", "Gold"),
    ("Windy City Cycles", "Dealer", "Midwest", "Illinois", "Generalist", "Silver"),
    ("River Bend Bikes", "Dealer", "Midwest", "Missouri", "Aggressive MTB", "Silver"),

    # Southeast
    ("Blue Ridge Cycles", "Dealer", "Southeast", "North Carolina", "Trail Shop", "Gold"),
    ("Pisgah Bike Co.", "Dealer", "Southeast", "North Carolina", "Aggressive MTB", "Platinum"),
    ("Southern Singletrack", "Dealer", "Southeast", "Georgia", "Generalist", "Silver"),
    ("Appalachian Bikes", "Dealer", "Southeast", "Tennessee", "Trail Shop", "Gold"),

    # Northeast
    ("Green Mountain Bikes", "Dealer", "Northeast", "Vermont", "XC Specialist", "Gold"),
    ("Adirondack Cycles", "Dealer", "Northeast", "New York", "Trail Shop", "Silver"),
    ("Pine Tree Bicycles", "Dealer", "Northeast", "Maine", "Generalist", "Silver"),
]

In [30]:
customer_rows = []

for i, dealer in enumerate(dealer_data, start=1):

    customer_rows.append({

        "customer_id": f"C{i:04}",

        "customer_name": dealer[0],

        "customer_type": dealer[1],

        "region": dealer[2],

        "state": dealer[3],

        "dealer_profile": dealer[4],

        "dealer_tier": dealer[5],

        "account_open_date": np.random.choice(
            pd.date_range("2018-01-01", "2023-12-31")
        ),

        "active_flag": True

    })

customers_df = pd.DataFrame(customer_rows)

In [31]:
customers_df.head()

,customer_id,customer_name,customer_type,region,state,dealer_profile,dealer_tier,account_open_date,active_flag
0,C0001,Sonoran Cycles Online,DTC,National,Online,Direct,N/A,2020-05-10,True
1,C0002,Desert Trail Bikes,Dealer,Southwest,Arizona,Trail Shop,Platinum,2021-07-18,True
2,C0003,Cactus Cycling,Dealer,Southwest,Arizona,Generalist,Gold,2021-02-04,True
3,C0004,Sonoran Bike Co.,Dealer,Southwest,Arizona,Aggressive MTB,Gold,2020-12-31,True
4,C0005,Mesa Mountain Bikes,Dealer,Southwest,Arizona,XC Specialist,Silver,2022-06-27,True


In [32]:
customers_df.shape

(36, 9)

In [33]:
customers_df.groupby("region").size()

region
Midwest       5
Mountain      7
National      1
Northeast     3
Southeast     4
Southwest     8
West Coast    8
dtype: int64

In [34]:
customers_df.to_csv(
    DATA_FOLDER / "customers.csv",
    index=False
)

print("customers.csv saved successfully!")

customers.csv saved successfully!


In [35]:
# -----------------------------
# GLOBAL CONFIGURATION
# -----------------------------

START_DATE = "2021-01-01"
END_DATE = "2025-12-31"

ORDER_ID_START = 100000

WAREHOUSE = "Phoenix DC"

SALES_ORDER_STATUS = [
    "Complete"
]

PAYMENT_TERMS = [
    "Net 30",
    "Net 45",
    "Credit Card"
]

In [36]:
# ---------------------------------
# SIZE DEMAND DISTRIBUTION
# ---------------------------------

size_weights = {
    "XS": 0.03,
    "S": 0.15,
    "M": 0.35,
    "L": 0.30,
    "XL": 0.15,
    "XXL": 0.02
}

In [37]:
seasonality = {

    1:0.75,

    2:1.15,

    3:1.40,

    4:1.60,

    5:1.70,

    6:1.45,

    7:1.25,

    8:1.10,

    9:1.00,

    10:0.90,

    11:1.20,

    12:0.80

}

In [38]:
year_growth = {

    2021:1.00,

    2022:1.08,

    2023:1.19,

    2024:1.27,

    2025:1.35

}

In [39]:
promotion_calendar = {

    (1,15):0.05,

    (2,1):0.08,

    (5,25):0.10,

    (11,25):0.18,

    (12,15):0.10

}

In [40]:
dealer_preferences = {

    "Trail Shop":{

        "Sabino":0.45,

        "Romero":0.30,

        "Oracle":0.15,

        "Catalina":0.05,

        "Rincon":0.03,

        "Sky Island":0.01,

        "Sonoita":0.01

    },

    "XC Specialist":{

        "Catalina":0.45,

        "Rincon":0.35,

        "Sabino":0.10,

        "Romero":0.05,

        "Oracle":0.03,

        "Sky Island":0.01,

        "Sonoita":0.01

    },

    "Aggressive MTB":{

        "Romero":0.45,

        "Oracle":0.35,

        "Sabino":0.15,

        "Catalina":0.02,

        "Rincon":0.01,

        "Sky Island":0.01,

        "Sonoita":0.01

    },

    "Premium Dealer":{

        "Sky Island":0.45,

        "Oracle":0.20,

        "Romero":0.15,

        "Sabino":0.10,

        "Catalina":0.05,

        "Rincon":0.03,

        "Sonoita":0.02

    },

    "Gravel Shop":{

        "Sonoita":0.70,

        "Catalina":0.10,

        "Sabino":0.08,

        "Rincon":0.05,

        "Romero":0.04,

        "Oracle":0.02,

        "Sky Island":0.01

    },

    "Generalist":{

        "Sabino":0.28,

        "Romero":0.20,

        "Catalina":0.18,

        "Oracle":0.15,

        "Sky Island":0.07,

        "Sonoita":0.07,

        "Rincon":0.05

    },

    "Direct":{

        "Sabino":0.30,

        "Catalina":0.20,

        "Romero":0.18,

        "Oracle":0.12,

        "Sky Island":0.08,

        "Sonoita":0.07,

        "Rincon":0.05

    }

}

In [41]:
# ---------------------------------
# LOAD MASTER DATA
# ---------------------------------

products_df = pd.read_csv(DATA_FOLDER / "products.csv")

customers_df = pd.read_csv(DATA_FOLDER / "customers.csv")

calendar_df = pd.read_csv(DATA_FOLDER / "calendar.csv")

products_df.head()

,product_id,model_name,category,travel_mm,wheel_size,frame_material,size,color,msrp,launch_date,active_flag
0,1001,Catalina,Cross Country,120,29,Carbon,XS,Ironwood,3299,2022-03-01,True
1,1002,Catalina,Cross Country,120,29,Carbon,XS,Saguaro,3299,2022-03-01,True
2,1003,Catalina,Cross Country,120,29,Carbon,XS,Sunset,3299,2022-03-01,True
3,1004,Catalina,Cross Country,120,29,Carbon,XS,Granite,3299,2022-03-01,True
4,1005,Catalina,Cross Country,120,29,Carbon,XS,Coyote,3299,2022-03-01,True


In [42]:
products_df.shape
customers_df.shape
calendar_df.shape

(1826, 15)

In [43]:
dealer_frequency = {

    "Platinum": 1.60,

    "Gold": 1.20,

    "Silver": 0.80,

    "Bronze": 0.50,

    "N/A": 1.00

}

In [44]:
def get_daily_order_count(current_date):

    month = current_date.month
    year = current_date.year

    seasonal = seasonality[month]

    growth = year_growth[year]

    base_orders = 40

    expected = base_orders * seasonal * growth

    noise = np.random.normal(0, 5)

    return max(8, round(expected + noise))

In [45]:
def choose_customer():

    weights = customers_df["dealer_tier"].map(dealer_frequency)

    weights = weights / weights.sum()

    return customers_df.sample(
        n=1,
        weights=weights
    ).iloc[0]

In [46]:
def choose_product(profile):

    # Pick bike model
    preferences = dealer_preferences[profile]

    model = np.random.choice(
        list(preferences.keys()),
        p=list(preferences.values())
    )

    # Pick size
    size = choose_size()

    # Pick color
    color = choose_color()

    # Find matching SKU
    matching = products_df[
        (products_df["model"] == model) &
        (products_df["size"] == size) &
        (products_df["color"] == color)
    ]

    return matching.iloc[0]

In [47]:
def choose_quantity(customer_type):

    if customer_type == "Dealer":

        return np.random.randint(2,7)

    return 1

In [48]:
def line_count(customer_type):

    if customer_type == "Dealer":

        return np.random.randint(2,6)

    return np.random.randint(1,3)

In [49]:
def choose_size():

    return np.random.choice(
        list(size_weights.keys()),
        p=list(size_weights.values())
    )

In [50]:
def choose_color():

    return np.random.choice(colors)

In [51]:
def calculate_discount(customer_type, current_date):

    discount = 0

    month = current_date.month

    day = current_date.day

    # Dealer preseason discounts
    if customer_type == "Dealer" and month in [1, 2]:
        discount = np.random.choice([0.05, 0.08, 0.10])

    # Memorial Day
    elif month == 5 and 20 <= day <= 31:
        discount = 0.10

    # Black Friday
    elif month == 11 and 22 <= day <= 30:
        discount = np.random.choice([0.15, 0.18, 0.20])

    # Holiday Sale
    elif month == 12 and day >= 15:
        discount = 0.10

    return discount

In [52]:
def calculate_price(product, discount):

    unit_price = product["msrp"] * (1 - discount)

    revenue = unit_price

    cost = product["standard_cost"]

    gross_profit = revenue - cost

    return (
        round(unit_price,2),
        round(cost,2),
        round(gross_profit,2)
    )

In [53]:
# ---------------------------------
# INITIAL INVENTORY
# ---------------------------------

inventory = {}

for _, row in products_df.iterrows():

    inventory[row["product_id"]] = {

        "on_hand": np.random.randint(12, 45),

        "allocated": 0

    }

In [54]:
def inventory_available(product_id):

    return inventory[product_id]["on_hand"]

In [55]:
def consume_inventory(product_id, quantity):

    inventory[product_id]["on_hand"] -= quantity

In [56]:
def inventory_check(product_id, requested_qty):

    available = inventory[product_id]["on_hand"]

    if available >= requested_qty:

        return requested_qty, "Filled"

    elif available > 0:

        return available, "Partial"

    else:

        return 0, "Backordered"

In [57]:
reorder_points = {}

for _, row in products_df.iterrows():

    reorder_points[row["product_id"]] = np.random.randint(8,15)

In [58]:
# ---------------------------------
# SUPPLIER MASTER
# ---------------------------------

supplier_data = [
    ("S001", "Fox Factory", "USA", 21),
    ("S002", "SRAM", "Taiwan", 30),
    ("S003", "Shimano", "Japan", 45),
    ("S004", "DT Swiss", "Switzerland", 28),
    ("S005", "Maxxis", "Taiwan", 35),
    ("S006", "Race Face", "Canada", 28),
    ("S007", "OneUp Components", "Canada", 24),
    ("S008", "Wolf Tooth", "USA", 18),
    ("S009", "Cane Creek", "USA", 20),
    ("S010", "Industry Nine", "USA", 16)
]

suppliers_df = pd.DataFrame(
    supplier_data,
    columns=[
        "supplier_id",
        "supplier_name",
        "country",
        "lead_time_days"
    ]
)

suppliers_df

supplier_ids = suppliers_df["supplier_id"].tolist()

supplier_lookup = (
    suppliers_df
    .set_index("supplier_id")
    .to_dict("index")
)

products_df["supplier_id"] = np.random.choice(
    supplier_ids,
    size=len(products_df)
)

suppliers_df.to_csv(
    DATA_FOLDER / "suppliers.csv",
    index=False
)

In [59]:
products_df["supplier_id"] = np.random.choice(
    supplier_ids,
    size=len(products_df)
)

In [60]:
# =====================================================
# SIMULATION ENGINE
# =====================================================

# ---------------------------------------
# OUTPUT TABLES
# ---------------------------------------

sales_order_headers = []

sales_order_lines = []

purchase_orders = []

inventory_snapshots = []

forecast_history = []

# ---------------------------------------
# ORDER NUMBER GENERATORS
# ---------------------------------------

next_sales_order = 100000
next_purchase_order = 500000


def get_next_sales_order():

    global next_sales_order

    order = f"SO{next_sales_order}"

    next_sales_order += 1

    return order


def get_next_purchase_order():

    global next_purchase_order

    po = f"PO{next_purchase_order}"

    next_purchase_order += 1

    return po

def receive_purchase_orders(current_date):

    global purchase_order_queue

    received = []

    for po in purchase_order_queue:

        if po["arrival_date"] <= current_date:

            inventory[po["product_id"]]["on_hand"] += po["quantity"]

            received.append(po)

    purchase_order_queue = [
        po for po in purchase_order_queue
        if po not in received
    ]
    
def already_on_order(product_id):

    for po in purchase_order_queue:

        if po["product_id"] == product_id:

            return True

    return False

def create_purchase_order(product_id,
                          quantity,
                          current_date):

    if already_on_order(product_id):

        return

    supplier = products_df.loc[
        products_df["product_id"] == product_id,
        "supplier_id"
    ].iloc[0]

    lead_time = supplier_lookup[supplier]["lead_time_days"]

    arrival = current_date + pd.Timedelta(days=lead_time)

    po = {

        "purchase_order_id":
            get_next_purchase_order(),

        "product_id":
            product_id,

        "supplier_id":
            supplier,

        "quantity":
            quantity,

        "order_date":
            current_date,

        "arrival_date":
            arrival,

        "status":
            "Open"

    }

    purchase_order_queue.append(po)

    purchase_orders.append(po)
    
def review_inventory(current_date):

    for product_id in inventory.keys():

        current = inventory[product_id]["on_hand"]

        reorder = reorder_points[product_id]

        if current <= reorder:

            create_purchase_order(

                product_id,

                quantity=50,

                current_date=current_date

            )
            
def capture_inventory(current_date):

    for product_id in inventory:

        inventory_snapshots.append({

            "date": current_date,

            "product_id": product_id,

            "on_hand":
                inventory[product_id]["on_hand"]

        })
        
def run_day(current_date):

    receive_purchase_orders(current_date)

    generate_daily_orders(current_date)

    review_inventory(current_date)

    capture_inventory(current_date)

In [61]:
model_supplier = {
    "Catalina": "S002",
    "Sabino": "S001",
    "Romero": "S001",
    "Oracle": "S001",
    "Sky Island": "S001",
    "Sonoita": "S002",
    "Rincon": "S002"
}

products_df["supplier_id"] = products_df["model_name"].map(model_supplier)

In [62]:
model_codes = {
    "Sabino": "SAB",
    "Romero": "ROM",
    "Catalina": "CAT",
    "Oracle": "ORA",
    "Sky Island": "SKY",
    "Rincon": "RIN",
    "Sonoita": "SON"
}

In [63]:
material_codes = {
    "Carbon": "C",
    "Aluminum": "A"
}

In [64]:
color_codes = {
    "Ironwood": "IR",
    "Saguaro": "SG",
    "Sunset": "SS",
    "Granite": "GR",
    "Coyote": "CY",
    "Sky": "SK"
}

In [65]:
def create_sku(row):

    return "-".join([
        "SON",
        model_codes[row["model_name"]],
        material_codes[row["frame_material"]],
        row["size"],
        color_codes[row["color"]]
    ])

products_df["sku"] = products_df.apply(create_sku, axis=1)

In [66]:
cost_pct = {
    "Cross Country": 0.53,
    "Downcountry": 0.55,
    "Trail": 0.56,
    "Aggressive Trail": 0.58,
    "Enduro": 0.60,
    "eMTB": 0.64,
    "Gravel": 0.52
}

In [67]:
def calculate_standard_cost(row):

    category = row["category"]

    if category not in cost_pct:
        raise ValueError(f"Unknown category: {category}")

    return round(row["msrp"] * cost_pct[category], 2)

products_df["standard_cost"] = products_df.apply(
    calculate_standard_cost,
    axis=1
)

In [68]:
products_df.to_csv(
    DATA_FOLDER / "products.csv",
    index=False
)

In [69]:
def assign_product_tier(msrp):
    if msrp < 2500:
        return "Entry"
    elif msrp < 4000:
        return "Mid"
    elif msrp < 6000:
        return "Premium"
    else:
        return "Flagship"

products_df["product_tier"] = products_df["msrp"].apply(assign_product_tier)

In [ ]:
# =====================================================
# SALES SIMULATION ENGINE
# =====================================================

tier_weights = {
    "Platinum": 12,
    "Gold": 8,
    "Silver": 4,
    "Bronze": 2,
    "DTC": 1
}

customers_df["order_weight"] = customers_df["tier"].map(tier_weights)

def select_dealer():

    dealers = customers_df[
        customers_df["tier"] != "DTC"
    ]

    return dealers.sample(
        n=1,
        weights="order_weight"
    ).iloc[0]

def select_dtc_customer():

    dtc = customers_df[
        customers_df["tier"] == "DTC"
    ]

    return dtc.sample(n=1).iloc[0]

category_weights = {

    "Trail":28,

    "Cross Country":18,

    "Downcountry":14,

    "Aggressive Trail":16,

    "Enduro":14,

    "Gravel":8,

    "eMTB":2

}

products_df["sales_weight"] = (
    products_df["category"]
        .map(category_weights)
)

def choose_product():

    return products_df.sample(
        n=1,
        weights="sales_weight"
    ).iloc[0]

import random

def generate_dealer_order():

    customer = select_dealer()

    line_count = random.randint(2, 6)

    order = []

    for _ in range(line_count):

        product = choose_product()

        qty = random.randint(1, 5)

        order.append({

            "customer_id": customer["customer_id"],

            "product_id": product["product_id"],

            "qty": qty

        })

    return order

def generate_dtc_order():

    customer = select_dtc_customer()

    product = choose_product()

    qty = random.choices(

        [1,2],

        weights=[95,5]

    )[0]

    return [{

        "customer_id": customer["customer_id"],

        "product_id": product["product_id"],

        "qty": qty

    }]

